
### Notebook to collate all individual photometry files into one final file



In [ ]:

""" loading the relevant packages """

from glob import glob
import os

from datetime import date

import pandas as pd
import numpy as np



### Defining some useful functions



In [ ]:

def _mkdir(newdir):
    """ Creates the required directory for the output files. """
    """works the way a good mkdir should :)
        - already exists, silently complete
        - regular file in the way, raise an exception
        - parent directory(ies) does not exist, make them as well
    """
    if os.path.isdir(newdir):
        pass
    elif os.path.isfile(newdir):
        raise OSError("a file with the same name as the desired " \
                      "dir, '%s', already exists." % newdir)
    else:
        head, tail = os.path.split(newdir)
        if head and not os.path.isdir(head):
            _mkdir(head)
        #print "_mkdir %s" % repr(newdir)
        if tail:
            os.mkdir(newdir)
    return newdir


In [ ]:

def fnu2ABmag(flux):
    # simple function to convert flux --> AB mag
    mag = -2.5 * np.log10(flux) + 23.90 # flux in micro-Jy
    return mag


def ABmag2fnu(mag):
    # simple function to convert AB mag --> flux
    flux = 10**((mag - 23.90) / (-2.5))
    return flux


def efnu2eABmag(flux, eflux):
    # simple function to convert flux error --> AB mag error
    emag = (2.5 / np.log(10)) * (eflux / flux)
    return emag


In [ ]:

def read_metadata():
    """ useful for reading in transient metadata """
    
    data = {}

    with open("metadata.txt", "r") as f:
        for line in f:
            line = line.strip()

            # Skip empty lines and headers
            if not line or line.startswith("#"):
                continue

            key, value = line.split(":", 1)

            data[key.strip()] = value.strip()

    transient_name = data["transient_name"]
    ra = data["RA"]
    dec = data["Dec"]

    print("transient_name, ra, dec")
    print(transient_name, ra, dec)
    
    return transient_name, ra, dec


In [ ]:

""" defining Pan-STARRS filter information """

PS_filters           = ["g",     "r",     "i",     "z",     "y",     "w"]
PS_filter_eff_centre = [4810.16, 6155.47, 7503.03, 8668.36, 9613.60, 5980.70]
PS_filter_eff_width  = [1053.08, 1252.41, 1206.62, 997.72,  638.98,  2561.73]

PS_filter_df = pd.DataFrame(
    {
        "Filter" : PS_filters,
        "Eff_centre" : PS_filter_eff_centre,
        "Eff_width" : PS_filter_eff_width,
    }
)


In [ ]:

""" reading in -- and combining -- all individual photometry dataframes """

photometry_filenames = sorted(glob("../Photometry/csv_files/*"))

mega_df = pd.DataFrame({})

for photometry_filename in photometry_filenames:

    if len(mega_df) == 0:
        mega_df = pd.read_csv(photometry_filename, sep=",")

    else:
        tmp_df = pd.read_csv(photometry_filename, sep=",")
        mega_df = pd.concat([mega_df, tmp_df])

mega_df.sort_values(by=["mjd", "filter"], ascending=True, inplace=True)

mega_df.reset_index(drop=True, inplace=True)

mega_df


In [ ]:

""" computing 3-sigma and 5-sigma upper limits (where necessary) """

mega_df["DetectionSignificance"] = mega_df["Flux(uJy)"] / mega_df["eFlux(uJy)"]

upper_lims_fivesigma_flux = []
upper_lims_threesigma_flux = []

for aa in range(len(mega_df)):
    tmp_df = mega_df.iloc[aa]

    sigma_cut = 3
    if tmp_df["DetectionSignificance"] < sigma_cut:
        flux_upper_lim = tmp_df["eFlux(uJy)"] * sigma_cut
    else:
        flux_upper_lim = "NaN"

    upper_lims_threesigma_flux.append(flux_upper_lim)


    sigma_cut = 5
    if tmp_df["DetectionSignificance"] < sigma_cut:
        flux_upper_lim = tmp_df["eFlux(uJy)"] * sigma_cut
    else:
        flux_upper_lim = "NaN"

    upper_lims_fivesigma_flux.append(flux_upper_lim)


mega_df["Flux_UL(3-sigma)"] = upper_lims_threesigma_flux
mega_df["Flux_UL(5-sigma)"] = upper_lims_fivesigma_flux

mega_df["Mag_UL(3-sigma)"] = fnu2ABmag(np.asarray(mega_df["Flux_UL(3-sigma)"]).astype(float))
mega_df["Mag_UL(5-sigma)"] = fnu2ABmag(np.asarray(mega_df["Flux_UL(5-sigma)"]).astype(float))

mega_df


In [ ]:

""" saving the final collated photometry file """

transient_name, ra, dec = read_metadata()

today = date.today()

mega_df.to_csv(f"{_mkdir('../FinalPhotometry/csv_files')}/TMP_{transient_name}_Pan-STARRS_photometry_{today}.csv", sep=",", index=False)

mega_df
